# Week 69, Day 5: Project: tests and profiling

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w069d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


### Exercise: Step 21: testing the tokeniser

The tokeniser is in `tiny_lex.py`. Write **pytest** tests (Week 24) that import `tokenize` and `LexError` from it, and pass on the correct version while catching each of six broken ones. Each broken version gets one detail wrong: two-character operators, the rule about a dot after a number, the values of `true`/`false`/`nil`, column numbering, comments, or escapes in strings. Think about what each of those should do, and test it. Use `pytest.raises` for at least one error.

In [ ]:
# Run this cell first: it creates the files used below.
with open('tiny_lex.py', 'w', encoding='utf-8') as f:
    f.write('from dataclasses import dataclass\n\n\nclass TinyError(Exception):\n    """Base class for every error the Tiny interpreter reports."""\n\n    def __init__(self, message, line=None, column=None):\n        super().__init__(message)\n        self.message, self.line, self.column = message, line, column\n\n    def __str__(self):\n        if self.line is None:\n            return self.message\n        if self.column is None:\n            return f"line {self.line}: {self.message}"\n        return f"line {self.line}, column {self.column}: {self.message}"\n\n\nclass LexError(TinyError):\n    """The source contains characters that don\'t form a token."""\n\n\nclass ParseError(TinyError):\n    """The tokens don\'t follow Tiny\'s grammar."""\n\n\nclass TinyRuntimeError(TinyError):\n    """Something went wrong while a program ran."""\n\n\n@dataclass(frozen=True)\nclass Token:\n    kind: str\n    text: str\n    value: object\n    line: int\n    column: int\n\n\nclass Cursor:\n    def __init__(self, source):\n        self.source, self.pos, self.line, self.column = source, 0, 1, 1\n\n    def at_end(self):\n        return self.pos >= len(self.source)\n\n    def peek(self, ahead=0):\n        i = self.pos + ahead\n        return self.source[i] if i < len(self.source) else ""\n\n    def advance(self):\n        ch = self.source[self.pos]\n        self.pos += 1\n        if ch == "\\n":\n            self.line, self.column = self.line + 1, 1\n        else:\n            self.column += 1\n        return ch\n\n\ndef read_number(cursor):\n    line, column, text = cursor.line, cursor.column, ""\n    while cursor.peek().isdigit():\n        text += cursor.advance()\n    if cursor.peek() == "." and cursor.peek(1).isdigit():\n        text += cursor.advance()\n        while cursor.peek().isdigit():\n            text += cursor.advance()\n        return Token("NUMBER", text, float(text), line, column)\n    return Token("NUMBER", text, int(text), line, column)\n\n\nESCAPES = {"n": "\\n", "t": "\\t", \'"\': \'"\', "\\\\": "\\\\"}\n\n\ndef read_string(cursor):\n    line, column = cursor.line, cursor.column\n    text = cursor.advance()\n    value = ""\n    while True:\n        if cursor.at_end() or cursor.peek() == "\\n":\n            raise LexError("unterminated string", line, column)\n        ch = cursor.advance()\n        text += ch\n        if ch == \'"\':\n            return Token("STRING", text, value, line, column)\n        if ch == "\\\\":\n            if cursor.at_end():\n                raise LexError("unterminated string", line, column)\n            esc_line, esc_col = cursor.line, cursor.column - 1\n            code = cursor.advance()\n            text += code\n            if code not in ESCAPES:\n                raise LexError(f"unknown escape \\\\{code}", esc_line, esc_col)\n            value += ESCAPES[code]\n        else:\n            value += ch\n\n\nKEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}\n\n\nTWO_CHAR = {"==", "!=", "<=", ">="}\n\n\nONE_CHAR = set("+-*/%<>=(){},;")\n\n\ndef read_name(cursor):\n    line, column, text = cursor.line, cursor.column, ""\n    while cursor.peek().isalnum() or cursor.peek() == "_":\n        text += cursor.advance()\n    if text in KEYWORDS:\n        value = {"true": True, "false": False, "nil": None}.get(text)\n        return Token(text.upper(), text, value, line, column)\n    return Token("NAME", text, text, line, column)\n\n\ndef read_operator(cursor):\n    line, column = cursor.line, cursor.column\n    pair = cursor.peek() + cursor.peek(1)\n    if pair in TWO_CHAR:\n        cursor.advance()\n        cursor.advance()\n        return Token("OP", pair, pair, line, column)\n    ch = cursor.peek()\n    if ch in ONE_CHAR:\n        cursor.advance()\n        return Token("OP", ch, ch, line, column)\n    raise LexError(f"unexpected character {ch!r}", line, column)\n\n\ndef skip_space_and_comments(cursor):\n    while not cursor.at_end():\n        ch = cursor.peek()\n        if ch in " \\t\\r\\n":\n            cursor.advance()\n        elif ch == "#":\n            while not cursor.at_end() and cursor.peek() != "\\n":\n                cursor.advance()\n        else:\n            return\n\n\ndef tokenize(source):\n    cursor = Cursor(source)\n    tokens = []\n    while True:\n        skip_space_and_comments(cursor)\n        if cursor.at_end():\n            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))\n            return tokens\n        ch = cursor.peek()\n        if ch.isdigit():\n            tokens.append(read_number(cursor))\n        elif ch == \'"\':\n            tokens.append(read_string(cursor))\n        elif ch.isalpha() or ch == "_":\n            tokens.append(read_name(cursor))\n        else:\n            tokens.append(read_operator(cursor))\n')

In [ ]:
import pytest
from tiny_lex import tokenize, LexError

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"tiny_lex.py\").read()\nBROKEN = {\n    \"two-character operators split\": GOOD.replace('TWO_CHAR = {\"==\", \"!=\", \"<=\", \">=\"}', 'TWO_CHAR = {\"==\", \"!=\"}'),\n    \"a dot always joins a number\": GOOD.replace('if cursor.peek() == \".\" and cursor.peek(1).isdigit():', 'if cursor.peek() == \".\":'),\n    \"keyword values lost\": GOOD.replace('value = {\"true\": True, \"false\": False, \"nil\": None}.get(text)', \"value = text\"),\n    \"columns counted from 0\": GOOD.replace(\"self.source, self.pos, self.line, self.column = source, 0, 1, 1\", \"self.source, self.pos, self.line, self.column = source, 0, 1, 0\"),\n    \"comments run to the end\": GOOD.replace('            while not cursor.at_end() and cursor.peek() != \"\\\\n\":\\n                cursor.advance()\\n        else:', '            while not cursor.at_end():\\n                cursor.advance()\\n        else:'),\n    \"escapes ignored\": GOOD.replace('ESCAPES = {\"n\": \"\\\\n\"', 'ESCAPES = {\"n\": \"n\"'),\n}\ndef test_versions_differ():\n    \"every broken version really is broken\"\n    assert all(v != GOOD for v in BROKEN.values())\ndef test_style():\n    \"pytest style\"\n    assert \"pytest.raises\" in source and \"def test_\" in source\ndef test_catches():\n    \"your tests pass on the correct tokeniser and catch every broken one\"\n    results = against(source, \"tiny_lex\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"tiny_lex.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Step 22: golden tests

Unit tests check pieces; **golden tests** check whole programs against their expected output, which catches problems in the way the pieces fit together. Write `check(cases)`, where each case is `(name, source, expected)`: `expected` is either the list of printed lines or, for a program that should fail, the string `"error: "` followed by the error's `str()`. Run each program with `run_source` and return a list of `(name, expected, got)` for the cases that don't match. Then `report(failures)`: `"all passed"`, or one line per failure, `FAIL name: expected ..., got ...` (with `repr` of both). The whole interpreter is given.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0

class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()

class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent

    def define(self, name, value):
        self.values[name] = value

    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None

    def get(self, name, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"undefined name {name!r}", line)
        return env.values[name]

    def assign(self, name, value, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)
        env.values[name] = value


class ReturnSignal(Exception):
    def __init__(self, value):
        self.value = value


class TinyFunction:
    def __init__(self, decl, closure):
        self.decl, self.closure = decl, closure

    def __repr__(self):
        return f"<fn {self.decl.name}>"


class Builtin:
    def __init__(self, name, arity, func):
        self.name, self.arity, self.func = name, arity, func

    def __repr__(self):
        return f"<builtin {self.name}>"


def truthy(value):
    return value is not None and value is not False


def show(value):
    if value is None:
        return "nil"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, float) and value.is_integer():
        return str(int(value)) if abs(value) < 1e16 else str(value)
    return str(value)


def type_name(value):
    if value is None:
        return "nil"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str):
        return "string"
    return "function"


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


class Interpreter:
    def __init__(self, output=None, max_depth=100):
        self.output = [] if output is None else output
        self.globals = Environment()
        self.max_depth, self.depth = max_depth, 0
        self.steps = 0
        for name, arity, func in [
            ("len", 1, self._len), ("str", 1, lambda v: show(v)), ("type", 1, type_name),
        ]:
            self.globals.define(name, Builtin(name, arity, func))

    @staticmethod
    def _len(value):
        if not isinstance(value, str):
            raise TinyRuntimeError(f"len() needs a string, not {type_name(value)}")
        return len(value)

    # --- expressions ---
    def evaluate(self, node, env):
        self.steps += 1
        if isinstance(node, (Number, String, Literal)):
            return node.value
        if isinstance(node, Name):
            return env.get(node.name, node.line)
        if isinstance(node, Unary):
            value = self.evaluate(node.operand, env)
            if node.op == "-":
                if not is_number(value):
                    raise TinyRuntimeError(f"cannot negate {type_name(value)}", node.line)
                return -value
            return not truthy(value)
        if isinstance(node, Logical):
            left = self.evaluate(node.left, env)
            if node.op == "or":
                return left if truthy(left) else self.evaluate(node.right, env)
            return left if not truthy(left) else self.evaluate(node.right, env)
        if isinstance(node, Binary):
            return self.binary(node, self.evaluate(node.left, env), self.evaluate(node.right, env))
        if isinstance(node, Call):
            return self.call(node, env)
        raise TinyRuntimeError(f"cannot evaluate {type(node).__name__}", getattr(node, "line", None))

    def binary(self, node, left, right):
        op = node.op
        if op == "==":
            return type_name(left) == type_name(right) and left == right
        if op == "!=":
            return not (type_name(left) == type_name(right) and left == right)
        if op == "+" and isinstance(left, str) and isinstance(right, str):
            return left + right
        if not (is_number(left) and is_number(right)):
            raise TinyRuntimeError(
                f"cannot use {op} with {type_name(left)} and {type_name(right)}", node.line)
        if op in ("/", "%") and right == 0:
            raise TinyRuntimeError("division by zero", node.line)
        if op == "/":
            result = left / right
            return int(result) if isinstance(left, int) and isinstance(right, int) and result.is_integer() else result
        return {"+": lambda: left + right, "-": lambda: left - right, "*": lambda: left * right,
                "%": lambda: left % right, "<": lambda: left < right, "<=": lambda: left <= right,
                ">": lambda: left > right, ">=": lambda: left >= right}[op]()

    def call(self, node, env):
        callee = self.evaluate(node.callee, env)
        args = [self.evaluate(a, env) for a in node.args]
        if isinstance(callee, Builtin):
            if len(args) != callee.arity:
                raise TinyRuntimeError(f"{callee.name}() takes {callee.arity} argument(s), got {len(args)}", node.line)
            try:
                return callee.func(*args)
            except TinyRuntimeError as e:
                raise TinyRuntimeError(e.message, node.line) from None
        if not isinstance(callee, TinyFunction):
            raise TinyRuntimeError(f"cannot call {type_name(callee)}", node.line)
        decl = callee.decl
        if len(args) != len(decl.params):
            raise TinyRuntimeError(f"{decl.name}() takes {len(decl.params)} argument(s), got {len(args)}", node.line)
        if self.depth >= self.max_depth:
            raise TinyRuntimeError("too much recursion", node.line)
        local = Environment(callee.closure)
        for name, value in zip(decl.params, args):
            local.define(name, value)
        self.depth += 1
        try:
            self.run_block(decl.body, local)
        except ReturnSignal as signal:
            return signal.value
        finally:
            self.depth -= 1
        return None

    # --- statements ---
    def execute(self, node, env):
        self.steps += 1
        if isinstance(node, Let):
            env.define(node.name, self.evaluate(node.value, env))
        elif isinstance(node, Assign):
            env.assign(node.name, self.evaluate(node.value, env), node.line)
        elif isinstance(node, Print):
            self.output.append(show(self.evaluate(node.value, env)))
        elif isinstance(node, ExprStmt):
            self.evaluate(node.value, env)
        elif isinstance(node, Block):
            self.run_block(node.body, Environment(env))
        elif isinstance(node, If):
            if truthy(self.evaluate(node.test, env)):
                self.execute(node.then, env)
            elif node.otherwise is not None:
                self.execute(node.otherwise, env)
        elif isinstance(node, While):
            while truthy(self.evaluate(node.test, env)):
                self.execute(node.body, env)
        elif isinstance(node, Function):
            env.define(node.name, TinyFunction(node, env))
        elif isinstance(node, Return):
            raise ReturnSignal(None if node.value is None else self.evaluate(node.value, env))
        else:
            raise TinyRuntimeError(f"cannot execute {type(node).__name__}", getattr(node, "line", None))

    def run_block(self, statements, env):
        for stmt in statements:
            self.execute(stmt, env)

    def run(self, program):
        try:
            self.run_block(program, self.globals)
        except ReturnSignal:
            raise TinyRuntimeError("return outside a function") from None
        return self.output


def run_source(source):
    interpreter = Interpreter()
    interpreter.run(parse(source))
    return interpreter.output

def show_error(source, error):
    if error.line is None:
        return str(error)
    text = source.split("\n")[error.line - 1]
    if error.column is None:
        return f"{error}\n    {text}"
    return f"{error}\n    {text}\n    {' ' * (error.column - 1)}^"


def check(cases):
    ___

def report(failures):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_golden():\n    \"golden tests\"\n    cases = [\n        (\"arithmetic\", \"print 1 + 2 * 3;\", [\"7\"]),\n        (\"closure\", \"fn f() { let n = 1; fn g() { return n; } return g; } print f()();\", [\"1\"]),\n        (\"type error\", 'print 1 + \"a\";', \"error: line 1: cannot use + with number and string\"),\n        (\"parse error\", \"print (1;\", \"error: line 1, column 9: expected ), got ';'\"),\n        (\"wrong\", \"print 2 + 2;\", [\"5\"]),\n    ]\n    failures = check(cases)\n    assert failures == [(\"wrong\", [\"5\"], [\"4\"])]\n    assert report(failures) == \"FAIL wrong: expected ['5'], got ['4']\"\n    assert report(check(cases[:4])) == \"all passed\"\n    assert check([(\"lex\", \"let a = 1 @ 2;\", \"error: line 1, column 11: unexpected character '@'\")]) == []")

### Exercise: Step 23: where the time goes

Profile the interpreter (Week 65). Write `CountingInterpreter`, a subclass of `Interpreter` that counts, in a `collections.Counter` called `counts`, how many times each kind of node (by class name) is evaluated or executed, then calls the parent method; `node_counts(source)` runs a program with it and returns the counter; `hottest(source, n=3)`: the `n` most common `(kind, count)` pairs, ties alphabetical; and `python_calls(source, method_name)`: run the program with a normal `Interpreter` under `cProfile` (parsing **before** profiling starts) and return how many calls were made to Python functions or methods with that name.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0

class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()

class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent

    def define(self, name, value):
        self.values[name] = value

    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None

    def get(self, name, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"undefined name {name!r}", line)
        return env.values[name]

    def assign(self, name, value, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)
        env.values[name] = value


class ReturnSignal(Exception):
    def __init__(self, value):
        self.value = value


class TinyFunction:
    def __init__(self, decl, closure):
        self.decl, self.closure = decl, closure

    def __repr__(self):
        return f"<fn {self.decl.name}>"


class Builtin:
    def __init__(self, name, arity, func):
        self.name, self.arity, self.func = name, arity, func

    def __repr__(self):
        return f"<builtin {self.name}>"


def truthy(value):
    return value is not None and value is not False


def show(value):
    if value is None:
        return "nil"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, float) and value.is_integer():
        return str(int(value)) if abs(value) < 1e16 else str(value)
    return str(value)


def type_name(value):
    if value is None:
        return "nil"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str):
        return "string"
    return "function"


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


class Interpreter:
    def __init__(self, output=None, max_depth=100):
        self.output = [] if output is None else output
        self.globals = Environment()
        self.max_depth, self.depth = max_depth, 0
        self.steps = 0
        for name, arity, func in [
            ("len", 1, self._len), ("str", 1, lambda v: show(v)), ("type", 1, type_name),
        ]:
            self.globals.define(name, Builtin(name, arity, func))

    @staticmethod
    def _len(value):
        if not isinstance(value, str):
            raise TinyRuntimeError(f"len() needs a string, not {type_name(value)}")
        return len(value)

    # --- expressions ---
    def evaluate(self, node, env):
        self.steps += 1
        if isinstance(node, (Number, String, Literal)):
            return node.value
        if isinstance(node, Name):
            return env.get(node.name, node.line)
        if isinstance(node, Unary):
            value = self.evaluate(node.operand, env)
            if node.op == "-":
                if not is_number(value):
                    raise TinyRuntimeError(f"cannot negate {type_name(value)}", node.line)
                return -value
            return not truthy(value)
        if isinstance(node, Logical):
            left = self.evaluate(node.left, env)
            if node.op == "or":
                return left if truthy(left) else self.evaluate(node.right, env)
            return left if not truthy(left) else self.evaluate(node.right, env)
        if isinstance(node, Binary):
            return self.binary(node, self.evaluate(node.left, env), self.evaluate(node.right, env))
        if isinstance(node, Call):
            return self.call(node, env)
        raise TinyRuntimeError(f"cannot evaluate {type(node).__name__}", getattr(node, "line", None))

    def binary(self, node, left, right):
        op = node.op
        if op == "==":
            return type_name(left) == type_name(right) and left == right
        if op == "!=":
            return not (type_name(left) == type_name(right) and left == right)
        if op == "+" and isinstance(left, str) and isinstance(right, str):
            return left + right
        if not (is_number(left) and is_number(right)):
            raise TinyRuntimeError(
                f"cannot use {op} with {type_name(left)} and {type_name(right)}", node.line)
        if op in ("/", "%") and right == 0:
            raise TinyRuntimeError("division by zero", node.line)
        if op == "/":
            result = left / right
            return int(result) if isinstance(left, int) and isinstance(right, int) and result.is_integer() else result
        return {"+": lambda: left + right, "-": lambda: left - right, "*": lambda: left * right,
                "%": lambda: left % right, "<": lambda: left < right, "<=": lambda: left <= right,
                ">": lambda: left > right, ">=": lambda: left >= right}[op]()

    def call(self, node, env):
        callee = self.evaluate(node.callee, env)
        args = [self.evaluate(a, env) for a in node.args]
        if isinstance(callee, Builtin):
            if len(args) != callee.arity:
                raise TinyRuntimeError(f"{callee.name}() takes {callee.arity} argument(s), got {len(args)}", node.line)
            try:
                return callee.func(*args)
            except TinyRuntimeError as e:
                raise TinyRuntimeError(e.message, node.line) from None
        if not isinstance(callee, TinyFunction):
            raise TinyRuntimeError(f"cannot call {type_name(callee)}", node.line)
        decl = callee.decl
        if len(args) != len(decl.params):
            raise TinyRuntimeError(f"{decl.name}() takes {len(decl.params)} argument(s), got {len(args)}", node.line)
        if self.depth >= self.max_depth:
            raise TinyRuntimeError("too much recursion", node.line)
        local = Environment(callee.closure)
        for name, value in zip(decl.params, args):
            local.define(name, value)
        self.depth += 1
        try:
            self.run_block(decl.body, local)
        except ReturnSignal as signal:
            return signal.value
        finally:
            self.depth -= 1
        return None

    # --- statements ---
    def execute(self, node, env):
        self.steps += 1
        if isinstance(node, Let):
            env.define(node.name, self.evaluate(node.value, env))
        elif isinstance(node, Assign):
            env.assign(node.name, self.evaluate(node.value, env), node.line)
        elif isinstance(node, Print):
            self.output.append(show(self.evaluate(node.value, env)))
        elif isinstance(node, ExprStmt):
            self.evaluate(node.value, env)
        elif isinstance(node, Block):
            self.run_block(node.body, Environment(env))
        elif isinstance(node, If):
            if truthy(self.evaluate(node.test, env)):
                self.execute(node.then, env)
            elif node.otherwise is not None:
                self.execute(node.otherwise, env)
        elif isinstance(node, While):
            while truthy(self.evaluate(node.test, env)):
                self.execute(node.body, env)
        elif isinstance(node, Function):
            env.define(node.name, TinyFunction(node, env))
        elif isinstance(node, Return):
            raise ReturnSignal(None if node.value is None else self.evaluate(node.value, env))
        else:
            raise TinyRuntimeError(f"cannot execute {type(node).__name__}", getattr(node, "line", None))

    def run_block(self, statements, env):
        for stmt in statements:
            self.execute(stmt, env)

    def run(self, program):
        try:
            self.run_block(program, self.globals)
        except ReturnSignal:
            raise TinyRuntimeError("return outside a function") from None
        return self.output


def run_source(source):
    interpreter = Interpreter()
    interpreter.run(parse(source))
    return interpreter.output

def show_error(source, error):
    if error.line is None:
        return str(error)
    text = source.split("\n")[error.line - 1]
    if error.column is None:
        return f"{error}\n    {text}"
    return f"{error}\n    {text}\n    {' ' * (error.column - 1)}^"


from collections import Counter
import cProfile
import pstats


class CountingInterpreter(Interpreter):
    ___

def node_counts(source):
    ___

def hottest(source, n=3):
    ___

def python_calls(source, method_name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FIB = 'fn fib(n) { if (n < 2) return n; return fib(n - 1) + fib(n - 2); } print fib(10);'\n\ndef test_profiling():\n    \"counting work\"\n    counts = node_counts(FIB)\n    assert counts[\"Call\"] == 177 and counts[\"Function\"] == 1 and counts[\"Print\"] == 1\n    assert hottest(FIB, 2) == [(\"Name\", counts[\"Name\"]), (\"Binary\", counts[\"Binary\"])]\n    assert counts[\"Name\"] > counts[\"Binary\"] > counts[\"Call\"]\n    assert node_counts(\"print 1;\") == {\"Print\": 1, \"Number\": 1}\n    calls = python_calls(FIB, \"call\")\n    assert calls == 177 and python_calls(FIB, \"binary\") == counts[\"Binary\"]\n    assert python_calls(\"print 1;\", \"tokenize\") == 0")

### Exercise: Step 24: a constant-folding pass

An optimiser can do some work before the program runs. Write `fold(node)`: return a copy of a tree (or a list of nodes) in which every `Binary` with two `Number` operands and an operator in `FOLDABLE` is replaced by its result (computed with the interpreter's own `binary`, so the rules match exactly; comparisons give a `Literal`), every `-` of a `Number` by a `Number`, and every `+` of two `String`s by a `String`, working **from the leaves up** so `2 * 3 + 1` folds completely. Don't fold `/` or `%` (which might divide by zero at run time, and must still raise then), and never change the original tree (use `dataclasses.replace`). Then `steps_saved(source)`: run the program with and without folding, check the outputs are equal, and return how many fewer interpreter steps the folded one took.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0

class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()

class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent

    def define(self, name, value):
        self.values[name] = value

    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None

    def get(self, name, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"undefined name {name!r}", line)
        return env.values[name]

    def assign(self, name, value, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)
        env.values[name] = value


class ReturnSignal(Exception):
    def __init__(self, value):
        self.value = value


class TinyFunction:
    def __init__(self, decl, closure):
        self.decl, self.closure = decl, closure

    def __repr__(self):
        return f"<fn {self.decl.name}>"


class Builtin:
    def __init__(self, name, arity, func):
        self.name, self.arity, self.func = name, arity, func

    def __repr__(self):
        return f"<builtin {self.name}>"


def truthy(value):
    return value is not None and value is not False


def show(value):
    if value is None:
        return "nil"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, float) and value.is_integer():
        return str(int(value)) if abs(value) < 1e16 else str(value)
    return str(value)


def type_name(value):
    if value is None:
        return "nil"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str):
        return "string"
    return "function"


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


class Interpreter:
    def __init__(self, output=None, max_depth=100):
        self.output = [] if output is None else output
        self.globals = Environment()
        self.max_depth, self.depth = max_depth, 0
        self.steps = 0
        for name, arity, func in [
            ("len", 1, self._len), ("str", 1, lambda v: show(v)), ("type", 1, type_name),
        ]:
            self.globals.define(name, Builtin(name, arity, func))

    @staticmethod
    def _len(value):
        if not isinstance(value, str):
            raise TinyRuntimeError(f"len() needs a string, not {type_name(value)}")
        return len(value)

    # --- expressions ---
    def evaluate(self, node, env):
        self.steps += 1
        if isinstance(node, (Number, String, Literal)):
            return node.value
        if isinstance(node, Name):
            return env.get(node.name, node.line)
        if isinstance(node, Unary):
            value = self.evaluate(node.operand, env)
            if node.op == "-":
                if not is_number(value):
                    raise TinyRuntimeError(f"cannot negate {type_name(value)}", node.line)
                return -value
            return not truthy(value)
        if isinstance(node, Logical):
            left = self.evaluate(node.left, env)
            if node.op == "or":
                return left if truthy(left) else self.evaluate(node.right, env)
            return left if not truthy(left) else self.evaluate(node.right, env)
        if isinstance(node, Binary):
            return self.binary(node, self.evaluate(node.left, env), self.evaluate(node.right, env))
        if isinstance(node, Call):
            return self.call(node, env)
        raise TinyRuntimeError(f"cannot evaluate {type(node).__name__}", getattr(node, "line", None))

    def binary(self, node, left, right):
        op = node.op
        if op == "==":
            return type_name(left) == type_name(right) and left == right
        if op == "!=":
            return not (type_name(left) == type_name(right) and left == right)
        if op == "+" and isinstance(left, str) and isinstance(right, str):
            return left + right
        if not (is_number(left) and is_number(right)):
            raise TinyRuntimeError(
                f"cannot use {op} with {type_name(left)} and {type_name(right)}", node.line)
        if op in ("/", "%") and right == 0:
            raise TinyRuntimeError("division by zero", node.line)
        if op == "/":
            result = left / right
            return int(result) if isinstance(left, int) and isinstance(right, int) and result.is_integer() else result
        return {"+": lambda: left + right, "-": lambda: left - right, "*": lambda: left * right,
                "%": lambda: left % right, "<": lambda: left < right, "<=": lambda: left <= right,
                ">": lambda: left > right, ">=": lambda: left >= right}[op]()

    def call(self, node, env):
        callee = self.evaluate(node.callee, env)
        args = [self.evaluate(a, env) for a in node.args]
        if isinstance(callee, Builtin):
            if len(args) != callee.arity:
                raise TinyRuntimeError(f"{callee.name}() takes {callee.arity} argument(s), got {len(args)}", node.line)
            try:
                return callee.func(*args)
            except TinyRuntimeError as e:
                raise TinyRuntimeError(e.message, node.line) from None
        if not isinstance(callee, TinyFunction):
            raise TinyRuntimeError(f"cannot call {type_name(callee)}", node.line)
        decl = callee.decl
        if len(args) != len(decl.params):
            raise TinyRuntimeError(f"{decl.name}() takes {len(decl.params)} argument(s), got {len(args)}", node.line)
        if self.depth >= self.max_depth:
            raise TinyRuntimeError("too much recursion", node.line)
        local = Environment(callee.closure)
        for name, value in zip(decl.params, args):
            local.define(name, value)
        self.depth += 1
        try:
            self.run_block(decl.body, local)
        except ReturnSignal as signal:
            return signal.value
        finally:
            self.depth -= 1
        return None

    # --- statements ---
    def execute(self, node, env):
        self.steps += 1
        if isinstance(node, Let):
            env.define(node.name, self.evaluate(node.value, env))
        elif isinstance(node, Assign):
            env.assign(node.name, self.evaluate(node.value, env), node.line)
        elif isinstance(node, Print):
            self.output.append(show(self.evaluate(node.value, env)))
        elif isinstance(node, ExprStmt):
            self.evaluate(node.value, env)
        elif isinstance(node, Block):
            self.run_block(node.body, Environment(env))
        elif isinstance(node, If):
            if truthy(self.evaluate(node.test, env)):
                self.execute(node.then, env)
            elif node.otherwise is not None:
                self.execute(node.otherwise, env)
        elif isinstance(node, While):
            while truthy(self.evaluate(node.test, env)):
                self.execute(node.body, env)
        elif isinstance(node, Function):
            env.define(node.name, TinyFunction(node, env))
        elif isinstance(node, Return):
            raise ReturnSignal(None if node.value is None else self.evaluate(node.value, env))
        else:
            raise TinyRuntimeError(f"cannot execute {type(node).__name__}", getattr(node, "line", None))

    def run_block(self, statements, env):
        for stmt in statements:
            self.execute(stmt, env)

    def run(self, program):
        try:
            self.run_block(program, self.globals)
        except ReturnSignal:
            raise TinyRuntimeError("return outside a function") from None
        return self.output


def run_source(source):
    interpreter = Interpreter()
    interpreter.run(parse(source))
    return interpreter.output

def show_error(source, error):
    if error.line is None:
        return str(error)
    text = source.split("\n")[error.line - 1]
    if error.column is None:
        return f"{error}\n    {text}"
    return f"{error}\n    {text}\n    {' ' * (error.column - 1)}^"


import dataclasses

FOLDABLE = {"+", "-", "*", "<", "<=", ">", ">=", "==", "!="}


def fold(node):
    ___

def steps_saved(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import dataclasses as _dc\n\ndef _shape(node):\n    if isinstance(node, list):\n        return [_shape(n) for n in node]\n    if _dc.is_dataclass(node):\n        return (type(node).__name__,) + tuple(_shape(getattr(node, f.name)) for f in _dc.fields(node) if f.name != \"line\")\n    return node\n\ndef test_fold():\n    \"constant folding\"\n    prog = parse('let x = 2 * 3 + 1; print -(4 - 1) * x; print \"a\" + \"b\" + \"c\"; print 1 < 2; print 6 / 0;')\n    folded = fold(prog)\n    assert _shape(folded[0]) == (\"Let\", \"x\", (\"Number\", 7))\n    assert _shape(folded[1]) == (\"Print\", (\"Binary\", \"*\", (\"Number\", -3), (\"Name\", \"x\")))\n    assert _shape(folded[2]) == (\"Print\", (\"String\", \"abc\")) and _shape(folded[3]) == (\"Print\", (\"Literal\", True))\n    assert _shape(folded[4]) == (\"Print\", (\"Binary\", \"/\", (\"Number\", 6), (\"Number\", 0)))\n    assert _shape(prog[0]) == (\"Let\", \"x\", (\"Binary\", \"+\", (\"Binary\", \"*\", (\"Number\", 2), (\"Number\", 3)), (\"Number\", 1)))\n    loop = \"let i = 0; let t = 0; while (i < 10) { t = t + 2 * 3 * 4; i = i + 1; } print t;\"\n    assert steps_saved(loop) == 40 and steps_saved(\"print 1;\") == 0\n    assert fold(parse('print 1 + \"a\";'))[0].value.op == \"+\"")

### Exercise: Step 25: an interactive session

Interpreters usually come with an interactive prompt that keeps its state between inputs (Python's REPL). Write `Session()` with one `Interpreter` that lasts for the whole session: `feed(text)` parses and runs one piece of Tiny code and returns the lines it printed; if it raises a `TinyError`, it returns a one-item list with `show_error(text, error)` instead, and any lines printed before the error in that input are dropped, but variables and functions defined earlier stay. `history` lists every input fed; `defined()` lists the names in the global scope (not the built-ins), sorted.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0

class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()

class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent

    def define(self, name, value):
        self.values[name] = value

    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None

    def get(self, name, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"undefined name {name!r}", line)
        return env.values[name]

    def assign(self, name, value, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)
        env.values[name] = value


class ReturnSignal(Exception):
    def __init__(self, value):
        self.value = value


class TinyFunction:
    def __init__(self, decl, closure):
        self.decl, self.closure = decl, closure

    def __repr__(self):
        return f"<fn {self.decl.name}>"


class Builtin:
    def __init__(self, name, arity, func):
        self.name, self.arity, self.func = name, arity, func

    def __repr__(self):
        return f"<builtin {self.name}>"


def truthy(value):
    return value is not None and value is not False


def show(value):
    if value is None:
        return "nil"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, float) and value.is_integer():
        return str(int(value)) if abs(value) < 1e16 else str(value)
    return str(value)


def type_name(value):
    if value is None:
        return "nil"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str):
        return "string"
    return "function"


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


class Interpreter:
    def __init__(self, output=None, max_depth=100):
        self.output = [] if output is None else output
        self.globals = Environment()
        self.max_depth, self.depth = max_depth, 0
        self.steps = 0
        for name, arity, func in [
            ("len", 1, self._len), ("str", 1, lambda v: show(v)), ("type", 1, type_name),
        ]:
            self.globals.define(name, Builtin(name, arity, func))

    @staticmethod
    def _len(value):
        if not isinstance(value, str):
            raise TinyRuntimeError(f"len() needs a string, not {type_name(value)}")
        return len(value)

    # --- expressions ---
    def evaluate(self, node, env):
        self.steps += 1
        if isinstance(node, (Number, String, Literal)):
            return node.value
        if isinstance(node, Name):
            return env.get(node.name, node.line)
        if isinstance(node, Unary):
            value = self.evaluate(node.operand, env)
            if node.op == "-":
                if not is_number(value):
                    raise TinyRuntimeError(f"cannot negate {type_name(value)}", node.line)
                return -value
            return not truthy(value)
        if isinstance(node, Logical):
            left = self.evaluate(node.left, env)
            if node.op == "or":
                return left if truthy(left) else self.evaluate(node.right, env)
            return left if not truthy(left) else self.evaluate(node.right, env)
        if isinstance(node, Binary):
            return self.binary(node, self.evaluate(node.left, env), self.evaluate(node.right, env))
        if isinstance(node, Call):
            return self.call(node, env)
        raise TinyRuntimeError(f"cannot evaluate {type(node).__name__}", getattr(node, "line", None))

    def binary(self, node, left, right):
        op = node.op
        if op == "==":
            return type_name(left) == type_name(right) and left == right
        if op == "!=":
            return not (type_name(left) == type_name(right) and left == right)
        if op == "+" and isinstance(left, str) and isinstance(right, str):
            return left + right
        if not (is_number(left) and is_number(right)):
            raise TinyRuntimeError(
                f"cannot use {op} with {type_name(left)} and {type_name(right)}", node.line)
        if op in ("/", "%") and right == 0:
            raise TinyRuntimeError("division by zero", node.line)
        if op == "/":
            result = left / right
            return int(result) if isinstance(left, int) and isinstance(right, int) and result.is_integer() else result
        return {"+": lambda: left + right, "-": lambda: left - right, "*": lambda: left * right,
                "%": lambda: left % right, "<": lambda: left < right, "<=": lambda: left <= right,
                ">": lambda: left > right, ">=": lambda: left >= right}[op]()

    def call(self, node, env):
        callee = self.evaluate(node.callee, env)
        args = [self.evaluate(a, env) for a in node.args]
        if isinstance(callee, Builtin):
            if len(args) != callee.arity:
                raise TinyRuntimeError(f"{callee.name}() takes {callee.arity} argument(s), got {len(args)}", node.line)
            try:
                return callee.func(*args)
            except TinyRuntimeError as e:
                raise TinyRuntimeError(e.message, node.line) from None
        if not isinstance(callee, TinyFunction):
            raise TinyRuntimeError(f"cannot call {type_name(callee)}", node.line)
        decl = callee.decl
        if len(args) != len(decl.params):
            raise TinyRuntimeError(f"{decl.name}() takes {len(decl.params)} argument(s), got {len(args)}", node.line)
        if self.depth >= self.max_depth:
            raise TinyRuntimeError("too much recursion", node.line)
        local = Environment(callee.closure)
        for name, value in zip(decl.params, args):
            local.define(name, value)
        self.depth += 1
        try:
            self.run_block(decl.body, local)
        except ReturnSignal as signal:
            return signal.value
        finally:
            self.depth -= 1
        return None

    # --- statements ---
    def execute(self, node, env):
        self.steps += 1
        if isinstance(node, Let):
            env.define(node.name, self.evaluate(node.value, env))
        elif isinstance(node, Assign):
            env.assign(node.name, self.evaluate(node.value, env), node.line)
        elif isinstance(node, Print):
            self.output.append(show(self.evaluate(node.value, env)))
        elif isinstance(node, ExprStmt):
            self.evaluate(node.value, env)
        elif isinstance(node, Block):
            self.run_block(node.body, Environment(env))
        elif isinstance(node, If):
            if truthy(self.evaluate(node.test, env)):
                self.execute(node.then, env)
            elif node.otherwise is not None:
                self.execute(node.otherwise, env)
        elif isinstance(node, While):
            while truthy(self.evaluate(node.test, env)):
                self.execute(node.body, env)
        elif isinstance(node, Function):
            env.define(node.name, TinyFunction(node, env))
        elif isinstance(node, Return):
            raise ReturnSignal(None if node.value is None else self.evaluate(node.value, env))
        else:
            raise TinyRuntimeError(f"cannot execute {type(node).__name__}", getattr(node, "line", None))

    def run_block(self, statements, env):
        for stmt in statements:
            self.execute(stmt, env)

    def run(self, program):
        try:
            self.run_block(program, self.globals)
        except ReturnSignal:
            raise TinyRuntimeError("return outside a function") from None
        return self.output


def run_source(source):
    interpreter = Interpreter()
    interpreter.run(parse(source))
    return interpreter.output

def show_error(source, error):
    if error.line is None:
        return str(error)
    text = source.split("\n")[error.line - 1]
    if error.column is None:
        return f"{error}\n    {text}"
    return f"{error}\n    {text}\n    {' ' * (error.column - 1)}^"


class Session:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_session():\n    \"a REPL\"\n    s = Session()\n    assert s.feed(\"let x = 2;\") == [] and s.feed(\"print x * 21;\") == [\"42\"]\n    assert s.feed(\"fn double(n) { return n * 2; }\") == [] and s.feed(\"print double(x);\") == [\"4\"]\n    error = s.feed('print 1; print x + \"a\";')\n    assert error == ['line 1: cannot use + with number and string\\n    print 1; print x + \"a\";']\n    assert s.feed(\"print x;\") == [\"2\"] and s.defined() == [\"double\", \"x\"]\n    bad = s.feed(\"let = 3;\")\n    assert bad[0].startswith(\"line 1, column 5: expected a name\") and bad[0].endswith(\"^\")\n    assert len(s.history) == 7 and Session().defined() == []")

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0

class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()

class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent

    def define(self, name, value):
        self.values[name] = value

    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None

    def get(self, name, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"undefined name {name!r}", line)
        return env.values[name]

    def assign(self, name, value, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)
        env.values[name] = value


class ReturnSignal(Exception):
    def __init__(self, value):
        self.value = value


class TinyFunction:
    def __init__(self, decl, closure):
        self.decl, self.closure = decl, closure

    def __repr__(self):
        return f"<fn {self.decl.name}>"


class Builtin:
    def __init__(self, name, arity, func):
        self.name, self.arity, self.func = name, arity, func

    def __repr__(self):
        return f"<builtin {self.name}>"


def truthy(value):
    return value is not None and value is not False


def show(value):
    if value is None:
        return "nil"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, float) and value.is_integer():
        return str(int(value)) if abs(value) < 1e16 else str(value)
    return str(value)


def type_name(value):
    if value is None:
        return "nil"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str):
        return "string"
    return "function"


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


class Interpreter:
    def __init__(self, output=None, max_depth=100):
        self.output = [] if output is None else output
        self.globals = Environment()
        self.max_depth, self.depth = max_depth, 0
        self.steps = 0
        for name, arity, func in [
            ("len", 1, self._len), ("str", 1, lambda v: show(v)), ("type", 1, type_name),
        ]:
            self.globals.define(name, Builtin(name, arity, func))

    @staticmethod
    def _len(value):
        if not isinstance(value, str):
            raise TinyRuntimeError(f"len() needs a string, not {type_name(value)}")
        return len(value)

    # --- expressions ---
    def evaluate(self, node, env):
        self.steps += 1
        if isinstance(node, (Number, String, Literal)):
            return node.value
        if isinstance(node, Name):
            return env.get(node.name, node.line)
        if isinstance(node, Unary):
            value = self.evaluate(node.operand, env)
            if node.op == "-":
                if not is_number(value):
                    raise TinyRuntimeError(f"cannot negate {type_name(value)}", node.line)
                return -value
            return not truthy(value)
        if isinstance(node, Logical):
            left = self.evaluate(node.left, env)
            if node.op == "or":
                return left if truthy(left) else self.evaluate(node.right, env)
            return left if not truthy(left) else self.evaluate(node.right, env)
        if isinstance(node, Binary):
            return self.binary(node, self.evaluate(node.left, env), self.evaluate(node.right, env))
        if isinstance(node, Call):
            return self.call(node, env)
        raise TinyRuntimeError(f"cannot evaluate {type(node).__name__}", getattr(node, "line", None))

    def binary(self, node, left, right):
        op = node.op
        if op == "==":
            return type_name(left) == type_name(right) and left == right
        if op == "!=":
            return not (type_name(left) == type_name(right) and left == right)
        if op == "+" and isinstance(left, str) and isinstance(right, str):
            return left + right
        if not (is_number(left) and is_number(right)):
            raise TinyRuntimeError(
                f"cannot use {op} with {type_name(left)} and {type_name(right)}", node.line)
        if op in ("/", "%") and right == 0:
            raise TinyRuntimeError("division by zero", node.line)
        if op == "/":
            result = left / right
            return int(result) if isinstance(left, int) and isinstance(right, int) and result.is_integer() else result
        return {"+": lambda: left + right, "-": lambda: left - right, "*": lambda: left * right,
                "%": lambda: left % right, "<": lambda: left < right, "<=": lambda: left <= right,
                ">": lambda: left > right, ">=": lambda: left >= right}[op]()

    def call(self, node, env):
        callee = self.evaluate(node.callee, env)
        args = [self.evaluate(a, env) for a in node.args]
        if isinstance(callee, Builtin):
            if len(args) != callee.arity:
                raise TinyRuntimeError(f"{callee.name}() takes {callee.arity} argument(s), got {len(args)}", node.line)
            try:
                return callee.func(*args)
            except TinyRuntimeError as e:
                raise TinyRuntimeError(e.message, node.line) from None
        if not isinstance(callee, TinyFunction):
            raise TinyRuntimeError(f"cannot call {type_name(callee)}", node.line)
        decl = callee.decl
        if len(args) != len(decl.params):
            raise TinyRuntimeError(f"{decl.name}() takes {len(decl.params)} argument(s), got {len(args)}", node.line)
        if self.depth >= self.max_depth:
            raise TinyRuntimeError("too much recursion", node.line)
        local = Environment(callee.closure)
        for name, value in zip(decl.params, args):
            local.define(name, value)
        self.depth += 1
        try:
            self.run_block(decl.body, local)
        except ReturnSignal as signal:
            return signal.value
        finally:
            self.depth -= 1
        return None

    # --- statements ---
    def execute(self, node, env):
        self.steps += 1
        if isinstance(node, Let):
            env.define(node.name, self.evaluate(node.value, env))
        elif isinstance(node, Assign):
            env.assign(node.name, self.evaluate(node.value, env), node.line)
        elif isinstance(node, Print):
            self.output.append(show(self.evaluate(node.value, env)))
        elif isinstance(node, ExprStmt):
            self.evaluate(node.value, env)
        elif isinstance(node, Block):
            self.run_block(node.body, Environment(env))
        elif isinstance(node, If):
            if truthy(self.evaluate(node.test, env)):
                self.execute(node.then, env)
            elif node.otherwise is not None:
                self.execute(node.otherwise, env)
        elif isinstance(node, While):
            while truthy(self.evaluate(node.test, env)):
                self.execute(node.body, env)
        elif isinstance(node, Function):
            env.define(node.name, TinyFunction(node, env))
        elif isinstance(node, Return):
            raise ReturnSignal(None if node.value is None else self.evaluate(node.value, env))
        else:
            raise TinyRuntimeError(f"cannot execute {type(node).__name__}", getattr(node, "line", None))

    def run_block(self, statements, env):
        for stmt in statements:
            self.execute(stmt, env)

    def run(self, program):
        try:
            self.run_block(program, self.globals)
        except ReturnSignal:
            raise TinyRuntimeError("return outside a function") from None
        return self.output


def run_source(source):
    interpreter = Interpreter()
    interpreter.run(parse(source))
    return interpreter.output

def show_error(source, error):
    if error.line is None:
        return str(error)
    text = source.split("\n")[error.line - 1]
    if error.column is None:
        return f"{error}\n    {text}"
    return f"{error}\n    {text}\n    {' ' * (error.column - 1)}^"


def check(cases):
    failures = []
    for name, src, expected in cases:
        try:
            got = run_source(src)
        except TinyError as e:
            got = "error: " + str(e)
        if got != expected:
            failures.append((name, expected, got))
    return failures


def report(failures):
    if not failures:
        return "all passed"
    lines = []
    for name, expected, got in failures:
        lines.append(f"FAIL {name}: expected {expected!r}, got {got!r}")
    return "\n".join(lines)


from collections import Counter
import cProfile
import pstats


class CountingInterpreter(Interpreter):
    def __init__(self):
        super().__init__()
        self.counts = Counter()

    def evaluate(self, node, env):
        self.counts[type(node).__name__] += 1
        return super().evaluate(node, env)

    def execute(self, node, env):
        self.counts[type(node).__name__] += 1
        return super().execute(node, env)


def node_counts(source):
    interp = CountingInterpreter()
    interp.run(parse(source))
    return interp.counts


def hottest(source, n=3):
    counts = node_counts(source)
    return sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:n]


def python_calls(source, method_name):
    profiler = cProfile.Profile()
    program = parse(source)
    interp = Interpreter()
    profiler.enable()
    try:
        interp.run(program)
    finally:
        profiler.disable()
    total = 0
    for (file, line, name), (primitive, calls, *rest) in pstats.Stats(profiler).stats.items():
        if name == method_name:
            total += calls
    return total


import dataclasses

FOLDABLE = {"+", "-", "*", "<", "<=", ">", ">=", "==", "!="}


def fold(node):
    if isinstance(node, list):
        return [fold(n) for n in node]
    if not dataclasses.is_dataclass(node):
        return node
    changes = {f.name: fold(getattr(node, f.name)) for f in dataclasses.fields(node) if f.name != "line"}
    node = dataclasses.replace(node, **changes)
    if isinstance(node, Unary) and node.op == "-" and isinstance(node.operand, Number):
        return Number(-node.operand.value, node.line)
    if isinstance(node, Binary) and node.op in FOLDABLE:
        left, right = node.left, node.right
        if isinstance(left, Number) and isinstance(right, Number):
            value = Interpreter().binary(node, left.value, right.value)
            return Literal(value, node.line) if isinstance(value, bool) else Number(value, node.line)
        if node.op == "+" and isinstance(left, String) and isinstance(right, String):
            return String(left.value + right.value, node.line)
    return node


def steps_saved(source):
    program = parse(source)
    plain, folded = Interpreter(), Interpreter()
    out_plain = plain.run(program)
    out_folded = folded.run(fold(program))
    assert out_plain == out_folded
    return plain.steps - folded.steps


cases = [("fib", 'fn fib(n) { if (n < 2) return n; return fib(n - 1) + fib(n - 2); } print fib(10);', ["55"]),
         ("strings", 'print "ab" + "cd";', ["abcd"]),
         ("error", "print undefined;", "error: line 1: undefined name 'undefined'")]
print(report(check(cases)))
print(hottest('fn fib(n) { if (n < 2) return n; return fib(n - 1) + fib(n - 2); } print fib(10);', 4))
print("folding saves", steps_saved("let i = 0; while (i < 5) { print 60 * 60 * 24 * i; i = i + 1; }"), "steps")

> **Tests make optimisation safe.** Step 24 changed the program the interpreter runs, and the only reason you can be confident it's correct is that Steps 21 and 22 gave you a way to check. That's the general rule: performance work without tests is guesswork, because an optimisation that changes behaviour in a corner case (folding a division by zero, say, or changing the order of an error) is a bug, however fast it is. Real compilers have test suites of tens of thousands of programs for exactly this reason.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

The interpreter is complete: `tiny_lex.py` (errors, tokens and the tokeniser), the syntax tree and parser, and the evaluator. Today's exercises give you all of it.

## Step 21: testing the tokeniser

Each broken version below is a mistake that's easy to make and easy to miss, and each one changes how some Tiny programs behave. A good test suite is one that would catch them.

## Step 22: golden tests

## Step 23: where the time goes

A tree-walking interpreter does a lot of Python work per Tiny operation: a method call, a chain of `isinstance` checks and a dictionary lookup for every node. Counting tells you which nodes matter.

## Step 24: a constant-folding pass

**Constant folding** is one of the oldest optimisations: compute at compile time whatever doesn't depend on the program's input. CPython does it too (Week 64: `2 * 3 + 1` is stored as `7` in the bytecode). The rule that matters most is that the optimised program must behave **exactly** like the original, errors included.

## Step 25: an interactive session

## Checkpoint

A golden-test run, a profile and the effect of folding, together:

## Quiz

## Before you go

Add a golden test of your own for a program that uses closures and recursion together, and check it passes. Tomorrow is the project review: the whole interpreter in one place, a code review, and one more feature.